# Budget-Limited Research Agent with LangGraph + TealTiger

This cookbook builds a multi-step research agent that keeps useful partial results when its cost budget is exhausted. It demonstrates:

- a LangGraph loop with separate search and summarization tools;
- `langchain-tealtiger` middleware for deterministic tool-call governance;
- TealTiger's public `CostTracker` and `BudgetManager` APIs for per-request and per-session cost control;
- `MONITOR` mode, which reports projected violations but lets the graph continue; and
- `ENFORCE` mode, which stops before an over-budget model call and returns partial research.

The default path is deterministic and does not need API keys. Set `TEALTIGER_LIVE_OPENAI=1` and `OPENAI_API_KEY` to use `ChatOpenAI`; the live path reads token usage from each response before recording actual cost. The deterministic path uses documented token fixtures so that budget exhaustion is reproducible.

## 1. Install dependencies

Run this cell once in a clean notebook environment. The versions below are the public APIs used by this example.

In [ ]:
%pip install -q "tealtiger==1.4.0" "langchain-tealtiger==0.1.0" "langgraph==1.2.11" "langchain-openai==1.6.0"

## 2. Configure a reproducible research workload

Each search costs `$0.03`. Each deterministic GPT-4 summarization uses a token fixture that TealTiger prices at `$0.45`. Five complete steps would cost `$2.40`, so a `$2.00` session budget allows four summaries and then blocks the fifth summarization in `ENFORCE` mode. The `$0.50` per-request limit is checked before every model call.

In [ ]:
from __future__ import annotations

import asyncio
import os
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from types import SimpleNamespace
from typing import Callable, Literal, TypedDict, cast

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tealtiger import TealTigerMiddleware
from langgraph.graph import END, START, StateGraph
from tealtiger import (
    BudgetManager,
    BudgetScope,
    CostBreakdown,
    CostRecord,
    CostTracker,
    TokenUsage,
)
from tealtiger.cost.storage import InMemoryCostStorage

AGENT_ID = "budget-research-agent"
PRICING_MODEL = "gpt-4"
LIVE_MODEL = os.getenv("TEALTIGER_DEMO_MODEL", PRICING_MODEL)
LIVE_OPENAI = (
    os.getenv("TEALTIGER_LIVE_OPENAI") == "1"
    and bool(os.getenv("OPENAI_API_KEY"))
)
SESSION_BUDGET_USD = 2.00
PER_REQUEST_LIMIT_USD = 0.50
MOCK_SEARCH_COST_USD = 0.03
MAX_OUTPUT_TOKENS = 1_024

# At GPT-4 prices in TealTiger 1.4.0, this fixture costs $0.45.
ESTIMATED_USAGE = TokenUsage(
    input_tokens=7_000,
    output_tokens=4_000,
    total_tokens=11_000,
)

RESEARCH_QUERIES = [
    "Why do autonomous research agents accumulate runaway cost?",
    "How should an agent estimate model cost before execution?",
    "What is the difference between monitoring and enforcement?",
    "How can an agent return partial results after a policy stop?",
    "Which cost metrics should production teams expose?",
]

print(f"Execution mode: {'live ChatOpenAI' if LIVE_OPENAI else 'deterministic fixture'}")

## 3. TealTiger cost-governance runtime

The middleware allowlists the two tools used by the graph. TealTiger's cost APIs estimate model spend from token usage, store actual records, and enforce an agent-scoped total budget. A separate preflight check enforces the per-request cap. All cost records include a `tool` label so the final report can break spend down by search versus summarization.

In [ ]:
class ResearchState(TypedDict):
    topic: str
    pending_queries: list[str]
    findings: list[dict[str, object]]
    summaries: list[str]
    governance_events: list[dict[str, object]]
    stopped_reason: str | None


class ResearchGovernanceRuntime:
    """Connect LangChain tool governance with TealTiger cost controls."""

    def __init__(self, mode: Literal["MONITOR", "ENFORCE"]) -> None:
        self.mode = mode
        self.middleware = TealTigerMiddleware(
            policies=[
                {
                    "type": "tool_allowlist",
                    "tools": ["mock_search", "summarize"],
                }
            ],
            agent_id=AGENT_ID,
            mode=mode,
        )
        self.middleware.before_agent({}, SimpleNamespace())
        self.model = (
            ChatOpenAI(
                model=LIVE_MODEL,
                temperature=0,
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            if LIVE_OPENAI
            else None
        )
        self.cost_tracker = CostTracker()
        self.storage = InMemoryCostStorage()
        self.budget_manager = BudgetManager(self.storage)
        self.session_budget = self.budget_manager.create_budget(
            name="research-session",
            limit=SESSION_BUDGET_USD,
            period="total",
            alert_thresholds=[50, 80, 100],
            action="block",
            scope=BudgetScope(type="agent", id=AGENT_ID),
        )

    def run_tool(
        self,
        name: str,
        args: dict[str, object],
        handler: Callable[[], object],
    ) -> tuple[bool, object]:
        """Execute the real tool handler inside the middleware boundary."""
        call_id = str(uuid.uuid4())
        request = SimpleNamespace(
            tool_call={"id": call_id, "name": name, "args": args}
        )
        result = self.middleware.wrap_tool_call(
            request,
            lambda _: handler(),
        )
        denied = (
            isinstance(result, ToolMessage)
            and str(result.content).startswith("[GOVERNANCE DENIED]")
        )
        return not denied, result

    async def preflight_cost(self, estimated_cost: float) -> dict[str, object]:
        request_violation = estimated_cost > PER_REQUEST_LIMIT_USD
        session_result = await self.budget_manager.check_budget(
            agent_id=AGENT_ID,
            estimated_cost=estimated_cost,
        )
        violation = request_violation or not session_result.allowed
        if request_violation:
            reason = (
                f"projected request cost ${estimated_cost:.2f} exceeds "
                f"the ${PER_REQUEST_LIMIT_USD:.2f} request limit"
            )
        elif not session_result.allowed:
            reason = (
                f"projected session cost exceeds the "
                f"${SESSION_BUDGET_USD:.2f} session budget"
            )
        else:
            reason = "within request and session budgets"

        return {
            "allowed": not violation or self.mode == "MONITOR",
            "action": "MONITOR" if violation and self.mode == "MONITOR" else (
                "DENY" if violation else "ALLOW"
            ),
            "reason": reason,
            "estimated_cost": estimated_cost,
        }

    async def record_search_cost(self, query: str) -> CostRecord:
        now = datetime.now(timezone.utc).replace(tzinfo=None).isoformat()
        zero_tokens = TokenUsage(input_tokens=0, output_tokens=0, total_tokens=0)
        record = CostRecord(
            id=str(uuid.uuid4()),
            request_id=str(uuid.uuid4()),
            agent_id=AGENT_ID,
            model="mock-search",
            provider="custom",
            actual_tokens=zero_tokens,
            actual_cost=MOCK_SEARCH_COST_USD,
            breakdown=CostBreakdown(
                input_cost=MOCK_SEARCH_COST_USD, output_cost=0.0
            ),
            timestamp=now,
            metadata={"tool": "mock_search", "query": query},
        )
        await self.storage.store(record)
        await self.budget_manager.record_cost(record)
        return record

    async def record_model_cost(
        self, query: str, usage: TokenUsage, model: str
    ) -> CostRecord:
        record = self.cost_tracker.calculate_actual_cost(
            request_id=str(uuid.uuid4()),
            agent_id=AGENT_ID,
            model=model,
            provider="openai",
            actual_tokens=usage,
            metadata={"tool": "summarize", "query": query},
        )
        # BudgetManager 1.4.0 uses naive UTC boundaries for total budgets.
        record = record.model_copy(
            update={
                "timestamp": datetime.now(timezone.utc)
                .replace(tzinfo=None)
                .isoformat()
            }
        )
        await self.storage.store(record)
        await self.budget_manager.record_cost(record)
        return record

    def finish(self) -> None:
        self.middleware.after_agent({}, SimpleNamespace())

    @property
    def total_cost(self) -> float:
        return sum(record.actual_cost for record in self.storage.records.values())

    def cost_by_tool(self) -> dict[str, float]:
        totals: defaultdict[str, float] = defaultdict(float)
        for record in self.storage.records.values():
            tool = str((record.metadata or {}).get("tool", "unknown"))
            totals[tool] += record.actual_cost
        return {tool: round(cost, 6) for tool, cost in totals.items()}

## 4. Search and summarize tools

The mock search keeps this notebook deterministic. The optional live summarizer reuses one capped `ChatOpenAI` client and reads `usage_metadata` from the returned message. Before a live call, the notebook estimates the real prompt tokens plus the configured maximum output; missing usage fails closed instead of being recorded as `$0`. The synchronous provider call runs in a worker thread so it does not block LangGraph's async event loop.

In [ ]:
MOCK_SOURCES = {
    query: [
        f"Source A for: {query}",
        f"Source B for: {query}",
    ]
    for query in RESEARCH_QUERIES
}


def mock_search(query: str) -> list[str]:
    return MOCK_SOURCES.get(query, [f"No fixture found for: {query}"])


def _usage_from_message(message: object) -> TokenUsage:
    usage = getattr(message, "usage_metadata", None) or {}
    if not usage:
        response_metadata = getattr(message, "response_metadata", {}) or {}
        usage = response_metadata.get("token_usage", {})
    if not usage:
        raise RuntimeError(
            "ChatOpenAI response omitted token usage; refusing to record $0"
        )
    input_tokens = int(usage.get("input_tokens", usage.get("prompt_tokens", 0)))
    output_tokens = int(
        usage.get("output_tokens", usage.get("completion_tokens", 0))
    )
    total_tokens = int(
        usage.get("total_tokens", input_tokens + output_tokens)
    )
    if total_tokens <= 0:
        raise RuntimeError("ChatOpenAI returned an empty token-usage record")
    return TokenUsage(
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
    )


def _summary_prompt(query: str, sources: list[str]) -> str:
    return (
        "Summarize the following research notes in two sentences.\n"
        f"Question: {query}\nNotes: {sources}"
    )


def estimate_summary_usage(
    runtime: ResearchGovernanceRuntime, prompt: str
) -> TokenUsage:
    if not LIVE_OPENAI:
        return ESTIMATED_USAGE
    if runtime.model is None:
        raise RuntimeError("Live mode requires a configured ChatOpenAI model")
    # Include a small framing buffer; output is capped by MAX_OUTPUT_TOKENS.
    input_tokens = runtime.model.get_num_tokens(prompt) + 64
    return TokenUsage(
        input_tokens=input_tokens,
        output_tokens=MAX_OUTPUT_TOKENS,
        total_tokens=input_tokens + MAX_OUTPUT_TOKENS,
    )


def summarize(
    runtime: ResearchGovernanceRuntime, query: str, sources: list[str]
) -> tuple[str, TokenUsage]:
    if not LIVE_OPENAI:
        summary = (
            f"Fixture synthesis for '{query}': agents should estimate spend "
            "before execution, retain evidence after a policy stop, and expose "
            "cost by step and tool."
        )
        return summary, ESTIMATED_USAGE

    if runtime.model is None:
        raise RuntimeError("Live mode requires a configured ChatOpenAI model")
    message = runtime.model.invoke(
        [HumanMessage(content=_summary_prompt(query, sources))]
    )
    return str(message.content), _usage_from_message(message)

## 5. Build the LangGraph loop

The graph performs one search and one summarization per query. Every paid operation is checked before execution, and each real tool handler executes inside `wrap_tool_call`, so middleware success and failure evidence describes the actual call. A denial or tool error sets `stopped_reason` and routes to `END`; findings already collected remain in state.

In [ ]:
def build_research_graph(runtime: ResearchGovernanceRuntime):
    async def search_step(state: ResearchState) -> dict[str, object]:
        query = state["pending_queries"][0]
        decision = await runtime.preflight_cost(MOCK_SEARCH_COST_USD)
        event = {"step": "search", "query": query, **decision}
        if not decision["allowed"]:
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(decision["reason"]),
            }

        try:
            allowed, tool_result = runtime.run_tool(
                "mock_search",
                {"query": query},
                lambda: mock_search(query),
            )
        except Exception as exc:
            event.update(action="ERROR", reason=f"mock_search failed: {exc}")
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(event["reason"]),
            }
        if not allowed:
            event.update(action="DENY", reason=str(tool_result.content))
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(event["reason"]),
            }

        sources = cast(list[str], tool_result)
        await runtime.record_search_cost(query)
        return {
            "findings": [*state["findings"], {"query": query, "sources": sources}],
            "governance_events": [*state["governance_events"], event],
        }

    async def summarize_step(state: ResearchState) -> dict[str, object]:
        finding = state["findings"][-1]
        query = str(finding["query"])
        sources = list(finding["sources"])
        prompt = _summary_prompt(query, sources)
        pricing_model = LIVE_MODEL if LIVE_OPENAI else PRICING_MODEL
        try:
            estimated_usage = estimate_summary_usage(runtime, prompt)
            estimate = runtime.cost_tracker.estimate_cost(
                model=pricing_model,
                provider="openai",
                estimated_tokens=estimated_usage,
            )
        except Exception as exc:
            event = {
                "step": "summarize",
                "query": query,
                "action": "ERROR",
                "reason": f"cost estimation failed: {exc}",
            }
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(event["reason"]),
            }

        decision = await runtime.preflight_cost(estimate.estimated_cost)
        event = {"step": "summarize", "query": query, **decision}
        if not decision["allowed"]:
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(decision["reason"]),
            }

        try:
            allowed, tool_result = await asyncio.to_thread(
                runtime.run_tool,
                "summarize",
                {"query": query},
                lambda: summarize(runtime, query, sources),
            )
        except Exception as exc:
            event.update(action="ERROR", reason=f"summarize failed: {exc}")
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(event["reason"]),
            }
        if not allowed:
            event.update(action="DENY", reason=str(tool_result.content))
            return {
                "governance_events": [*state["governance_events"], event],
                "stopped_reason": str(event["reason"]),
            }

        summary, usage = cast(tuple[str, TokenUsage], tool_result)
        record = await runtime.record_model_cost(query, usage, pricing_model)
        event["actual_cost"] = record.actual_cost
        return {
            "pending_queries": state["pending_queries"][1:],
            "summaries": [*state["summaries"], summary],
            "governance_events": [*state["governance_events"], event],
        }

    def after_search(state: ResearchState) -> Literal["summarize", "done"]:
        return "done" if state.get("stopped_reason") else "summarize"

    def after_summary(state: ResearchState) -> Literal["search", "done"]:
        if state.get("stopped_reason") or not state["pending_queries"]:
            return "done"
        return "search"

    graph = StateGraph(ResearchState)
    graph.add_node("search", search_step)
    graph.add_node("summarize", summarize_step)
    graph.add_edge(START, "search")
    graph.add_conditional_edges(
        "search", after_search, {"summarize": "summarize", "done": END}
    )
    graph.add_conditional_edges(
        "summarize", after_summary, {"search": "search", "done": END}
    )
    return graph.compile()


async def run_research(mode: Literal["MONITOR", "ENFORCE"]):
    runtime = ResearchGovernanceRuntime(mode)
    graph = build_research_graph(runtime)
    initial_state: ResearchState = {
        "topic": "Cost governance for autonomous research agents",
        "pending_queries": list(RESEARCH_QUERIES),
        "findings": [],
        "summaries": [],
        "governance_events": [],
        "stopped_reason": None,
    }
    if not initial_state["pending_queries"]:
        runtime.finish()
        return initial_state, runtime
    try:
        result = await graph.ainvoke(
            initial_state,
            config={"recursion_limit": 2 * len(RESEARCH_QUERIES) + 5},
        )
    finally:
        runtime.finish()
    return result, runtime

## 6. Compare MONITOR and ENFORCE

`MONITOR` records the fifth summarization as a projected violation and continues, finishing above budget. `ENFORCE` stops before that call, while preserving the fifth query's raw search finding and the four summaries already produced.

In [ ]:
monitor_state, monitor_runtime = await run_research("MONITOR")
enforce_state, enforce_runtime = await run_research("ENFORCE")


def print_report(label: str, state: ResearchState, runtime: ResearchGovernanceRuntime):
    print(f"\n{label}")
    print("-" * len(label))
    print(f"Summaries returned: {len(state['summaries'])}")
    print(f"Raw findings retained: {len(state['findings'])}")
    print(f"Total cost: ${runtime.total_cost:.2f}")
    print(f"Cost by tool: {runtime.cost_by_tool()}")
    print(f"Middleware evaluations: {len(runtime.middleware.evidence)}")
    print(f"Stopped reason: {state.get('stopped_reason')}")
    violations = [
        event for event in state["governance_events"]
        if event["action"] in {"MONITOR", "DENY"}
    ]
    print(f"Projected violations: {violations}")


print_report("MONITOR mode", monitor_state, monitor_runtime)
print_report("ENFORCE mode", enforce_state, enforce_runtime)

## 7. Verify the governance behavior

These assertions make the intended behavior executable rather than relying only on printed output.

In [ ]:
assert monitor_runtime.middleware.evidence
assert enforce_runtime.middleware.evidence

if not LIVE_OPENAI:
    assert len(monitor_state["summaries"]) == len(RESEARCH_QUERIES)
    assert monitor_runtime.total_cost > SESSION_BUDGET_USD
    assert any(
        event["action"] == "MONITOR"
        for event in monitor_state["governance_events"]
    )

    assert len(enforce_state["summaries"]) == len(RESEARCH_QUERIES) - 1
    assert len(enforce_state["findings"]) == len(RESEARCH_QUERIES)
    assert enforce_runtime.total_cost <= SESSION_BUDGET_USD
    assert enforce_state["stopped_reason"] is not None
    assert any(
        event["action"] == "DENY"
        for event in enforce_state["governance_events"]
    )
    assert not any(
        event["action"] == "ERROR"
        for event in monitor_state["governance_events"]
        + enforce_state["governance_events"]
    )
    print("Deterministic governance checks passed.")
else:
    summary_records = [
        record
        for runtime in (monitor_runtime, enforce_runtime)
        for record in runtime.storage.records.values()
        if (record.metadata or {}).get("tool") == "summarize"
    ]
    assert all(
        record.actual_cost <= PER_REQUEST_LIMIT_USD
        for record in summary_records
    )
    assert enforce_runtime.total_cost <= SESSION_BUDGET_USD
    print("Live safety invariants passed.")

print("All applicable governance checks passed.")

## Production notes

- Use provider response metadata as the source of actual token usage; fail closed when it is missing, and do not treat the deterministic fixture as billing data.
- Bound model output and reserve a conservative prompt-plus-output estimate before live calls. For strict billing guarantees across retries and concurrent workers, also enforce limits at the provider or proxy layer.
- Store cost records in persistent storage when budgets must span processes or replicas.
- Reserve estimated cost before a call and reconcile it with actual cost afterward to avoid concurrent requests overspending the same remaining budget.
- Keep partial findings in graph state or a checkpoint store so a governance stop still returns useful work.
- Start in `MONITOR`, inspect projected denials, then move to `ENFORCE` once pricing and thresholds are validated for your workload.